In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print("Ready")

Ready


In [2]:
%%writefile agents/fact_checking_agent.py
import json
import re
from typing import Dict, List

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "fact_checking_agent"
MAX_FACTS_PER_BATCH = 15  # keeps the prompt a reasonable size; chunk larger sets


def _build_fact_check_prompt(facts: List[Dict]) -> str:
    numbered = "\n".join(
        f'{i}. "{f["claim"]}" (sources: {len(f.get("sources", []))})'
        for i, f in enumerate(facts)
    )
    return f"""You are fact-checking a set of claims gathered during research. For EACH
numbered claim below, classify it as exactly one of:

- CONFIRMED: corroborated by multiple independent sources, or clearly consistent
  with other claims in this list
- DISPUTED: contradicts another claim in this list
- UNVERIFIED: plausible but only backed by a single source, with no corroboration
  or contradiction from the others

Claims:
{numbered}

Respond with ONLY a JSON array, one object per claim, in the same order:
[{{"index": 0, "status": "CONFIRMED", "reasoning": "one short sentence why"}}, ...]
No markdown, no explanation outside the JSON.
"""


def _parse_fact_check_response(raw_text: str, expected_count: int) -> List[Dict]:
    cleaned = re.sub(r"^```(json)?", "", raw_text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        parsed = json.loads(cleaned)
        if not isinstance(parsed, list):
            return []
        results = {}
        for item in parsed:
            idx = item.get("index")
            if isinstance(idx, int) and 0 <= idx < expected_count:
                status = item.get("status", "UNVERIFIED").upper()
                if status not in ("CONFIRMED", "DISPUTED", "UNVERIFIED"):
                    status = "UNVERIFIED"
                results[idx] = {"status": status, "reasoning": item.get("reasoning", "")}
        return [results.get(i, {"status": "UNVERIFIED", "reasoning": "not returned by model"})
                for i in range(expected_count)]
    except json.JSONDecodeError:
        return []


def fact_check_batch(facts: List[Dict]) -> List[Dict]:
    """Fact-checks a batch of facts. Never raises: on any failure, every fact in
    the batch is marked UNVERIFIED (fail-safe, never fail-open to CONFIRMED)."""
    if not facts:
        return []

    try:
        raw_output = call_llm(_build_fact_check_prompt(facts))
        verdicts = _parse_fact_check_response(raw_output, len(facts))
        if not verdicts:
            raise ValueError("fact-check response could not be parsed")
    except Exception as e:
        log_agent_event(AGENT_NAME, "batch_failed", error=f"{type(e).__name__}: {e}",
                        batch_size=len(facts))
        verdicts = [{"status": "UNVERIFIED", "reasoning": "fact-check call failed; defaulted to unverified"}
                   for _ in facts]

    checked = []
    for fact, verdict in zip(facts, verdicts):
        checked.append({**fact, "status": verdict["status"], "reasoning": verdict["reasoning"]})

    counts = {}
    for v in verdicts:
        counts[v["status"]] = counts.get(v["status"], 0) + 1
    log_agent_event(AGENT_NAME, "batch_done", **counts)
    return checked


def fact_check_facts(facts: List[Dict]) -> List[Dict]:
    """Chunks facts into batches and fact-checks each. Public entry point."""
    all_checked = []
    for i in range(0, len(facts), MAX_FACTS_PER_BATCH):
        batch = facts[i:i + MAX_FACTS_PER_BATCH]
        all_checked.extend(fact_check_batch(batch))
    return all_checked


def fact_checking_node(state: Dict) -> Dict:
    """LangGraph node: fact-checks every extracted fact in state."""
    extracted_facts = state.get("extracted_facts", [])
    if not extracted_facts:
        return {"errors": [{"agent": AGENT_NAME, "error": "no extracted_facts to check"}]}

    checked = fact_check_facts(extracted_facts)

    return {
        "fact_check_results": checked,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq",
                            "facts_checked": len(checked)}],
    }

Writing agents/fact_checking_agent.py


In [3]:
from agents.web_search_agent import search_one
from agents.data_extraction_agent import extract_facts_from_result
from agents.fact_checking_agent import fact_check_facts

search_result = search_one("What are the main safety risks of solid-state EV batteries?")
facts = extract_facts_from_result(search_result)
checked = fact_check_facts(facts)

for c in checked[:8]:
    print(f"[{c['status']:10}] {c['claim']}")
    print(f"             → {c['reasoning']}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


 Gemini unavailable (ClientError), falling back to Tavily...


2026-10-02 23:08:03,362 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3
2026-10-02 23:08:07,385 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the main safety risks of solid-state EV batteries? facts_found=14
2026-10-02 23:08:08,598 | INFO | research_system | [fact_checking_agent] batch_done CONFIRMED=13 UNVERIFIED=1


[CONFIRMED ] Continuous charging and discharging cycles lead to a gradual decline in battery capacity.
             → Consistent with general battery degradation principles and supported by multiple sources.
[CONFIRMED ] Frequent rapid charging accelerates the decline in battery capacity.
             → Consistent with claim 0 regarding capacity decline and supported by multiple sources.
[CONFIRMED ] Liquid electrolytes introduce risks of leakage.
             → Consistent with claim 5 regarding flammability risks and supported by multiple sources.
[CONFIRMED ] Liquid electrolytes introduce risks of dendrite formation.
             → Consistent with claim 4 describing dendrite characteristics and supported by multiple sources.
[CONFIRMED ] Dendrite formation is characterized by needle-like structures.
             → Consistent with claim 3 regarding dendrite formation and supported by multiple sources.
[CONFIRMED ] Removing flammable liquid electrolytes can reduce thermal runaway risk.

In [4]:
contradictory_facts = [
    {"claim": "Solid-state batteries have a lower fire risk than liquid-electrolyte batteries.", "sources": [{"url": "http://a.com"}]},
    {"claim": "Solid-state batteries have the same fire risk as liquid-electrolyte batteries.", "sources": [{"url": "http://b.com"}]},
    {"claim": "Tesla's headquarters is in Austin, Texas.", "sources": [{"url": "http://c.com"}, {"url": "http://d.com"}]},
]

checked = fact_check_facts(contradictory_facts)
for c in checked:
    print(f"[{c['status']:10}] {c['claim']}")
    print(f"             → {c['reasoning']}")

2026-10-02 23:20:17,254 | INFO | research_system | [fact_checking_agent] batch_done DISPUTED=2 UNVERIFIED=1


[DISPUTED  ] Solid-state batteries have a lower fire risk than liquid-electrolyte batteries.
             → This claim directly contradicts claim 1, which states the fire risk is the same.
[DISPUTED  ] Solid-state batteries have the same fire risk as liquid-electrolyte batteries.
             → This claim directly contradicts claim 0, which states the fire risk is lower.
[UNVERIFIED] Tesla's headquarters is in Austin, Texas.
             → This claim is plausible but is supported by only a single source with no corroboration or contradiction from other claims in the list.


In [5]:
import agents.fact_checking_agent as fca

original = fca.call_llm
fca.call_llm = lambda p: (_ for _ in ()).throw(RuntimeError("simulated API outage"))

checked = fca.fact_check_facts([{"claim": "Test claim during simulated outage.", "sources": []}])
print(checked)
assert checked[0]["status"] == "UNVERIFIED", "FAIL-SAFE BROKEN: must default to UNVERIFIED, never CONFIRMED"
print(" Fail-safe confirmed: failure defaults to UNVERIFIED, not CONFIRMED")

fca.call_llm = original  

2026-10-02 23:20:59,796 | INFO | research_system | [fact_checking_agent] batch_failed error=RuntimeError: simulated API outage batch_size=1
2026-10-02 23:20:59,797 | INFO | research_system | [fact_checking_agent] batch_done UNVERIFIED=1


[{'claim': 'Test claim during simulated outage.', 'sources': [], 'status': 'UNVERIFIED', 'reasoning': 'fact-check call failed; defaulted to unverified'}]
 Fail-safe confirmed: failure defaults to UNVERIFIED, not CONFIRMED


In [6]:
from agents.fact_checking_agent import fact_checking_node

state = {"extracted_facts": facts} 
out = fact_checking_node(state)
print(f"Checked {len(out['fact_check_results'])} facts")
status_counts = {}
for r in out["fact_check_results"]:
    status_counts[r["status"]] = status_counts.get(r["status"], 0) + 1
print("Status breakdown:", status_counts)

2026-10-02 23:22:14,541 | INFO | research_system | [fact_checking_agent] batch_done CONFIRMED=13 UNVERIFIED=1


Checked 14 facts
Status breakdown: {'CONFIRMED': 13, 'UNVERIFIED': 1}


In [7]:
%%writefile tests/test_fact_checking_agent.py
import agents.fact_checking_agent as fca


def test_batch_parses_valid_response(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: json.dumps([
        {"index": 0, "status": "CONFIRMED", "reasoning": "two sources agree"},
        {"index": 1, "status": "DISPUTED", "reasoning": "contradicts claim 0"},
    ]))
    facts = [{"claim": "A", "sources": []}, {"claim": "B", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "CONFIRMED"
    assert checked[1]["status"] == "DISPUTED"


def test_batch_defaults_to_unverified_on_llm_failure(monkeypatch):
    def boom(p):
        raise RuntimeError("down")
    monkeypatch.setattr(fca, "call_llm", boom)
    facts = [{"claim": "A", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "UNVERIFIED"


def test_batch_defaults_to_unverified_on_malformed_json(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: "not json")
    facts = [{"claim": "A", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "UNVERIFIED"


def test_batch_fills_missing_index_as_unverified(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: json.dumps([
        {"index": 0, "status": "CONFIRMED", "reasoning": "ok"}
    ]))
    facts = [{"claim": "A", "sources": []}, {"claim": "B", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "CONFIRMED"
    assert checked[1]["status"] == "UNVERIFIED"


def test_empty_facts_returns_empty(monkeypatch):
    assert fca.fact_check_facts([]) == []


def test_chunking_splits_large_batches(monkeypatch):
    calls = {"count": 0}
    def fake_call(p):
        calls["count"] += 1
        n = p.count('"')  
        return json.dumps([{"index": i, "status": "UNVERIFIED", "reasoning": "x"}
                           for i in range(fca.MAX_FACTS_PER_BATCH)])
    monkeypatch.setattr(fca, "call_llm", fake_call)
    facts = [{"claim": f"fact {i}", "sources": []} for i in range(fca.MAX_FACTS_PER_BATCH + 5)]
    checked = fca.fact_check_facts(facts)
    assert len(checked) == fca.MAX_FACTS_PER_BATCH + 5
    assert calls["count"] == 2


def test_node_empty_extracted_facts(monkeypatch):
    out = fca.fact_checking_node({"extracted_facts": []})
    assert out["errors"] and "fact_check_results" not in out

Writing tests/test_fact_checking_agent.py


In [ ]:
%%writefile tests/test_fact_checking_agent.py
import json
import agents.fact_checking_agent as fca


def test_batch_parses_valid_response(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: json.dumps([
        {"index": 0, "status": "CONFIRMED", "reasoning": "two sources agree"},
        {"index": 1, "status": "DISPUTED", "reasoning": "contradicts claim 0"},
    ]))
    facts = [{"claim": "A", "sources": []}, {"claim": "B", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "CONFIRMED"
    assert checked[1]["status"] == "DISPUTED"


def test_batch_defaults_to_unverified_on_llm_failure(monkeypatch):
    def boom(p):
        raise RuntimeError("down")
    monkeypatch.setattr(fca, "call_llm", boom)
    facts = [{"claim": "A", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "UNVERIFIED"


def test_batch_defaults_to_unverified_on_malformed_json(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: "not json")
    facts = [{"claim": "A", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "UNVERIFIED"


def test_batch_fills_missing_index_as_unverified(monkeypatch):
    monkeypatch.setattr(fca, "call_llm", lambda p: json.dumps([
        {"index": 0, "status": "CONFIRMED", "reasoning": "ok"}
    ]))
    facts = [{"claim": "A", "sources": []}, {"claim": "B", "sources": []}]
    checked = fca.fact_check_batch(facts)
    assert checked[0]["status"] == "CONFIRMED"
    assert checked[1]["status"] == "UNVERIFIED"


def test_empty_facts_returns_empty(monkeypatch):
    assert fca.fact_check_facts([]) == []


def test_chunking_splits_large_batches(monkeypatch):
    calls = {"count": 0}
    def fake_call(p):
        calls["count"] += 1
        return json.dumps([{"index": i, "status": "UNVERIFIED", "reasoning": "x"}
                           for i in range(fca.MAX_FACTS_PER_BATCH)])
    monkeypatch.setattr(fca, "call_llm", fake_call)
    facts = [{"claim": f"fact {i}", "sources": []} for i in range(fca.MAX_FACTS_PER_BATCH + 5)]
    checked = fca.fact_check_facts(facts)
    assert len(checked) == fca.MAX_FACTS_PER_BATCH + 5
    assert calls["count"] == 2


def test_node_empty_extracted_facts(monkeypatch):
    out = fca.fact_checking_node({"extracted_facts": []})
    assert out["errors"] and "fact_check_results" not in out